In [2]:
:dep base64 = "0.22"
:dep ureq = { version = "2", features = ["tls"] }
:dep scraper = "0.20"

In [10]:
use scraper::{Html, Selector};
use base64::{engine::general_purpose::STANDARD, Engine as _};
use std::fs;

/// 問題ページを GET して Base64 文字列を返す。
/// 問題文は <div class="bg-light border rounded p-3 mb-2"> 内にある。
fn fetch_challenge_text(url: &str) -> Result<String, Box<dyn std::error::Error>> {
    let html = ureq::get(url).call()?.into_string()?;
    let document = Html::parse_document(&html);
    // Bootstrap クラスが付いた div を狙い撃ち
    let selector = Selector::parse("div.bg-light.border.rounded").unwrap();
    let div = document
        .select(&selector)
        .next()
        .ok_or("対象の div タグが見つかりません")?;
    // テキストノードを結合し、空白・改行を全て除去
    let raw: String = div.text().collect::<Vec<_>>().join("");
    let cleaned: String = raw.chars().filter(|c| !c.is_whitespace()).collect();
    Ok(cleaned)
}

let url = "https://ksnctf.sweetduet.info/problem/5";
let b64_text = fetch_challenge_text(url)?;
println!("取得した文字列の先頭 60 文字: {}", &b64_text[..60.min(b64_text.len())]);
println!("全体の文字数: {}", b64_text.len());

/// Base64 デコードを繰り返し、uuencode ヘッダが現れたら停止する。
/// 戻り値: (uuencode テキスト, 総デコード回数)
fn peel_onion(initial: &str) -> Result<(String, usize), Box<dyn std::error::Error>> {
    let mut data: Vec<u8> = initial.as_bytes().to_vec();
    let mut count = 0usize;

    loop {
        // "begin" が含まれていれば uuencode データに到達
        if data.windows(5).any(|w| w == b"begin") {
            let text = String::from_utf8(data)?;
            return Ok((text, count));
        }

        // Base64 デコード — 改行・空白を除去してから処理
        let clean: Vec<u8> = data
            .iter()
            .copied()
            .filter(|b| !b.is_ascii_whitespace())
            .collect();

        data = STANDARD.decode(&clean).map_err(|e| {
            format!("Base64 デコード失敗 (ループ {}): {}", count + 1, e)
        })?;

        count += 1;
        println!("  decode #{:>2} — {} bytes", count, data.len());

        // 安全のための上限（問題は 16 回）
        if count > 100 {
            return Err("デコード上限 (100) に達しました".into());
        }
    }
}

println!("Base64 デコードを開始します…\n");
let (uu_text, decode_count) = peel_onion(&b64_text)?;

println!("\nBase64 デコード完了: {} 回", decode_count);
println!("--- uuencode データ ---");
println!("{}", uu_text);

/// uuencode テキストをデコードし、生バイト列を返す
fn uudecode(input: &str) -> Result<Vec<u8>, Box<dyn std::error::Error>> {
    let mut output: Vec<u8> = Vec::new();
    let mut in_body = false;

    for line in input.lines() {
        let line = line.trim_end_matches('\r'); // Windows 改行対応

        if line.starts_with("begin ") {
            in_body = true;
            continue;
        }
        if line == "end" {
            break;
        }
        if !in_body || line.is_empty() {
            continue;
        }

        let bytes = line.as_bytes();
        if bytes.is_empty() {
            continue;
        }

        // 行の先頭バイト = エンコード前のバイト数
        let length = ((bytes[0].wrapping_sub(b' ')) & 0x3F) as usize;
        if length == 0 {
            continue; // 空行（通常 " " 一文字だけの行）
        }

        // 6 bit × 4 = 3 byte のグループを処理
        let mut decoded_this_line: Vec<u8> = Vec::new();
        let mut i = 1usize;
        while i + 3 < bytes.len() {
            let a = (bytes[i    ].wrapping_sub(b' ')) & 0x3F;
            let b = (bytes[i + 1].wrapping_sub(b' ')) & 0x3F;
            let c = (bytes[i + 2].wrapping_sub(b' ')) & 0x3F;
            let d = (bytes[i + 3].wrapping_sub(b' ')) & 0x3F;

            decoded_this_line.push((a << 2) | (b >> 4));
            decoded_this_line.push((b << 4) | (c >> 2));
            decoded_this_line.push((c << 6) |  d      );
            i += 4;
        }

        // length バイト分だけ採用（パディング分を除く）
        decoded_this_line.truncate(length);
        output.extend_from_slice(&decoded_this_line);
    }

    Ok(output)
}

let flag_bytes = uudecode(&uu_text)?;
let flag = String::from_utf8(flag_bytes.clone())
    .unwrap_or_else(|_| format!("{:?}", flag_bytes));

println!("uuencode デコード完了 ({} bytes)", flag_bytes.len());

// flag.txt に書き出す（末尾改行付き）
fs::write("flag.txt", format!("{}", flag.trim()))?;

println!("FLAG を flag.txt に書き出しました。");
println!("(内容は flag.txt を直接確認してください)");

取得した文字列の先頭 60 文字: Vm0wd2QyUXlVWGxWV0d4V1YwZDRWMVl3WkRSV01WbDNXa1JTV0ZKdGVGWlZN
全体の文字数: 7112
Base64 デコードを開始します…

  decode # 1 — 5334 bytes
  decode # 2 — 3948 bytes
  decode # 3 — 2922 bytes
  decode # 4 — 2161 bytes
  decode # 5 — 1597 bytes
  decode # 6 — 1180 bytes
  decode # 7 — 872 bytes
  decode # 8 — 645 bytes
  decode # 9 — 475 bytes
  decode #10 — 349 bytes
  decode #11 — 256 bytes
  decode #12 — 187 bytes
  decode #13 — 138 bytes
  decode #14 — 102 bytes
  decode #15 — 73 bytes
  decode #16 — 53 bytes

Base64 デコード完了: 16 回
--- uuencode データ ---
begin 666 <data>
51DQ!1U]&94QG4#-3:4%797I74$AU
 
end

uuencode デコード完了 (21 bytes)
FLAG を flag.txt に書き出しました。
(内容は flag.txt を直接確認してください)
